# Module 3.4 — Higher-Order Schemes and TVD Limiters

**The problem you've been ignoring since Module 1.6:**
The upwind scheme is stable but diffusive — it smears sharp features. Central schemes are accurate but oscillate near discontinuities. In real CFD (combustion fronts, shocks, steep concentration gradients), you need both: second-order accuracy in smooth regions AND no oscillations near sharp features.

**Godunov's theorem (1959):** No *linear* scheme can be both second-order accurate and free of spurious oscillations (monotone). This is a mathematical theorem, not a deficiency of specific schemes — it is a fundamental limit.

**The solution:** use *nonlinear* schemes that adapt their behaviour based on the local gradient. These are called **flux limiters** or **TVD (Total Variation Diminishing) schemes**.

**Roadmap:**
1. The diffusion-dispersion dilemma — why no linear scheme works near discontinuities
2. Total Variation — what it measures and why decreasing TV is the right property
3. The flux limiter framework — how to blend upwind and high-order in one formula
4. Common limiters — Minmod, Van Leer, Superbee, MC
5. The Sweby diagram — a graphical classification of all TVD schemes
6. QUICK scheme — third-order accurate but not TVD
7. Implementation and comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The Diffusion-Dispersion Dilemma

### Modified equation analysis reveals the hidden physics

Every finite-difference scheme actually solves a **modified PDE** — not the original one, but one with extra derivative terms. The extra terms reveal the scheme's character:

**Upwind (FTBS)** actually solves:
$$\frac{\partial u}{\partial t} + c\frac{\partial u}{\partial x} = \underbrace{\frac{c\Delta x}{2}(1-\nu)\frac{\partial^2 u}{\partial x^2}}_{\text{numerical diffusion}}$$

The extra term is **diffusion** — it smears everything. At CFL $\nu = 1$, diffusion vanishes (upwind is exact). At $\nu < 1$, everything gets smeared.

**Lax-Wendroff** actually solves:
$$\frac{\partial u}{\partial t} + c\frac{\partial u}{\partial x} = \underbrace{-\frac{c\Delta x^2}{6}(1-\nu^2)\frac{\partial^3 u}{\partial x^3}}_{\text{numerical dispersion}}$$

The extra term is a **third-derivative** (dispersion) — different wave components travel at different speeds, creating oscillations.

### The consequence

- Diffusion: peaks get shorter, valleys fill in. Smooth, but features are lost.
- Dispersion: waves break up into oscillations that weren't in the original data — physically wrong and potentially catastrophic (negative concentrations, pressures).

Neither is acceptable for: combustion simulations (steep concentration fronts), shock capturing (discontinuous solutions), scalar transport with sharp interfaces.

## 2. Total Variation — Measuring Oscillations

### Definition

The **total variation** of a discrete solution $\{u_i\}$:

$$TV(u) = \sum_i |u_{i+1} - u_i|$$

For a smooth, monotone solution: $TV = |u_N - u_0|$ — just the total change.

For an oscillating solution: each oscillation adds to the sum. $TV$ grows with oscillation amplitude.

### TVD property

A scheme is **TVD** (Total Variation Diminishing) if:

$$TV(u^{n+1}) \leq TV(u^n) \quad \text{for all } n$$

**What this guarantees:** the solution cannot develop new local extrema that weren't in the initial condition. Overshoots and undershoots cannot appear. Godunov's theorem: no *linear* scheme is both TVD and second-order — the limiter function must be nonlinear.

## 3. The Flux Limiter Framework

### The blending formula

Write the face flux as a blend of upwind (low-order, stable) and high-order (accurate):

$$F_{i+1/2} = F^{\text{upwind}}_{i+1/2} + \phi(r_i)\left(F^{\text{high}}_{i+1/2} - F^{\text{upwind}}_{i+1/2}\right)$$

- $\phi(r) \in [0,1]$: the **limiter function** — 0 means pure upwind, 1 means pure high-order
- $r_i$: the **gradient ratio** — measures local smoothness:

$$r_i = \frac{u_i - u_{i-1}}{u_{i+1} - u_i}$$

**Why the gradient ratio?**
- $r \approx 1$: gradients are equal on both sides → smooth region → use high-order ($\phi = 1$)
- $r < 0$: local extremum (sign change in gradient) → near a peak or discontinuity → use upwind ($\phi = 0$)
- $r \gg 1$ or $r \ll 0$: strong non-uniformity → gradually reduce to upwind

## 4. Common Limiter Functions

| Limiter | $\phi(r)$ | Character |
|---------|-----------|----------|
| **Minmod** | $\max(0, \min(1, r))$ | Most diffusive TVD — safest choice |
| **Van Leer** | $(r+|r|)/(1+|r|)$ | Smooth curve, second-order, popular |
| **Superbee** | $\max(0, \min(2r,1), \min(r,2))$ | Least diffusive TVD — sharpest features |
| **MC** (monotonized central) | $\max(0,\min((1+r)/2, 2, 2r))$ | Good all-round balance |

All limiters satisfy $\phi(1) = 1$ (second-order in smooth regions) and $\phi(r) = 0$ for $r < 0$ (first-order near extrema).

In [ ]:
# ── Limiter functions and the Sweby diagram ───────────────────────────────────

r = np.linspace(-0.5, 3.5, 500)

limiters = {
    'Minmod':    np.maximum(0, np.minimum(1, r)),
    'Van Leer':  (r + np.abs(r)) / (1 + np.abs(r) + 1e-12),
    'Superbee':  np.maximum(0, np.maximum(np.minimum(2*r, 1), np.minimum(r, 2))),
    'MC':        np.maximum(0, np.minimum((1+r)/2, np.minimum(2, 2*r))),
    'QUICK (not TVD)': 0.75 + 0.25*r,   # can exceed Sweby region
}
colors = ['blue', 'red', 'purple', 'darkorange', 'gray']
styles = ['-', '-', '-', '-', '--']

# Sweby TVD region bounds
upper_bound = np.maximum(0, np.maximum(np.minimum(2*r, 1), np.minimum(r, 2)))
lower_bound = np.zeros_like(r)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ── Sweby diagram ─────────────────────────────────────────────────────────────
ax1.fill_between(r, lower_bound, upper_bound, alpha=0.15, color='green',
                  label='Sweby TVD region')
for (name, phi), col, sty in zip(limiters.items(), colors, styles):
    ax1.plot(r, phi, color=col, lw=2, linestyle=sty, label=name)

# Mark second-order locus: φ(1) = 1
ax1.axvline(1, color='k', lw=0.5, linestyle=':')
ax1.plot(1, 1, 'k*', markersize=12, label='$\\phi(1)=1$: 2nd-order point')
ax1.axhline(0, color='k', lw=0.5); ax1.axvline(0, color='k', lw=0.5)
ax1.set_xlabel('$r$ (gradient ratio)', fontsize=12)
ax1.set_ylabel('$\\phi(r)$ (limiter value)', fontsize=12)
ax1.set_title('Sweby diagram\nAll TVD limiters must lie within the green region')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_xlim(-0.5, 3.5); ax1.set_ylim(-0.1, 2.5)
ax1.text(1.5, 1.8, 'QUICK lies outside\nTVD region →\ncan oscillate', fontsize=9, color='gray')

# ── Interpretation of r ───────────────────────────────────────────────────────
ax2.axis('off')
cases_r = [
    ('$r < 0$', 'Local extremum\n(sign change in gradient)', '$\\phi = 0$: pure upwind', 'tomato'),
    ('$r \\approx 1$', 'Smooth, uniform gradient', '$\\phi = 1$: full 2nd-order', 'seagreen'),
    ('$r > 1$', 'Gradient steeper ahead\n(compression)', '$\\phi < 1$: partial limiter', 'steelblue'),
]
y_pos = 0.85
for r_label, description, phi_val, color in cases_r:
    ax2.text(0.05, y_pos, r_label, fontsize=14, color=color, fontweight='bold',
             transform=ax2.transAxes)
    ax2.text(0.30, y_pos, description, fontsize=10, transform=ax2.transAxes, va='center')
    ax2.text(0.75, y_pos, phi_val, fontsize=10, color=color, transform=ax2.transAxes, va='center')
    y_pos -= 0.22

ax2.set_title('Interpretation of gradient ratio $r$\nand corresponding limiter action', fontsize=11)
ax2.add_patch(plt.Rectangle((0,0), 1, 1, fill=False, transform=ax2.transAxes, clip_on=False))

plt.tight_layout()
plt.show()

print('Key insight: limiters are NONLINEAR — φ(r) depends on the local solution.')
print('This is what allows second-order accuracy in smooth regions + TVD everywhere.')
print('Linear schemes cannot achieve both simultaneously (Godunov theorem).')

In [ ]:
# ── Scheme comparison on a mixed test (step + Gaussian) ──────────────────────

def advect_tvd(u0, c, dx, dt, nt, scheme='upwind'):
    """1D advection with various flux treatments."""
    u  = u0.copy()
    nu = c * dt / dx

    def phi(r, name):
        if name == 'minmod':
            return np.maximum(0, np.minimum(1, r))
        elif name == 'van_leer':
            return (r + np.abs(r)) / (1 + np.abs(r) + 1e-12)
        elif name == 'superbee':
            return np.maximum(0, np.maximum(np.minimum(2*r,1), np.minimum(r,2)))
        elif name == 'mc':
            return np.maximum(0, np.minimum((1+r)/2, np.minimum(2, 2*r)))

    for _ in range(nt):
        un = u.copy()

        if scheme == 'upwind':
            u[1:] = un[1:] - nu*(un[1:] - un[:-1])
            u[0]  = un[0]  - nu*(un[0]  - un[-1])

        elif scheme == 'lax_wendroff':
            u[1:-1] = (un[1:-1]
                       - nu/2*(un[2:]-un[:-2])
                       + nu**2/2*(un[2:]-2*un[1:-1]+un[:-2]))
            u[0] = un[0]; u[-1] = un[-1]

        elif scheme in ('minmod', 'van_leer', 'superbee', 'mc'):
            # Flux-limited scheme (c > 0: upwind = left cell)
            u_new = np.zeros_like(un)
            N     = len(un)
            for i in range(N):
                im1 = (i-1) % N;  im2 = (i-2) % N
                ip1 = (i+1) % N
                denom = un[i] - un[im1] + 1e-12
                r_i   = (un[im1] - un[im2]) / denom
                phi_i = phi(r_i, scheme)
                denom2 = un[ip1] - un[i] + 1e-12
                r_im1  = (un[i] - un[im1]) / denom2
                phi_im1 = phi(r_im1, scheme)
                F_right = c*(un[i]   + 0.5*phi_i  *(1-nu)*(un[i]-un[im1]))
                F_left  = c*(un[im1] + 0.5*phi_im1*(1-nu)*(un[im1]-un[im2]))
                u_new[i] = un[i] - nu*(F_right - F_left)/c
            u = u_new
    return u

# ── Setup ─────────────────────────────────────────────────────────────────────
N   = 150
L   = 3.0;   dx = L/N
x   = np.linspace(0, L, N, endpoint=False)
c   = 1.0;   dt = 0.6*dx/c;   T = 1.5
nt  = int(T/dt)

# Mixed IC: step (hard) + Gaussian (smooth)
u0 = np.zeros(N)
u0[(x>0.2)&(x<0.5)] = 1.0
u0 += 0.8*np.exp(-((x-1.5)**2)/0.01)

# Exact: shift by c*T mod L
x_ex = (x - c*T) % L
u_ex = np.zeros(N)
u_ex[(x_ex>0.2)&(x_ex<0.5)] = 1.0
u_ex += 0.8*np.exp(-((x_ex-1.5)**2)/0.01)

schemes = ['upwind', 'lax_wendroff', 'minmod', 'van_leer', 'superbee']
results = {s: advect_tvd(u0, c, dx, dt, nt, s) for s in schemes}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes_flat = axes.ravel()

for ax, scheme in zip(axes_flat, schemes):
    u_num = results[scheme]
    l2 = np.sqrt(np.mean((u_num-u_ex)**2))
    tv = np.sum(np.abs(np.diff(u_num)))
    ax.plot(x, u0, 'k--', lw=1, alpha=0.3, label='Initial')
    ax.plot(x, u_ex, 'k-', lw=2, label='Exact')
    ax.plot(x, u_num, 'r-', lw=1.8,
            label=f'{scheme.replace("_"," ").title()}')
    ax.set_title(f'{scheme.replace("_"," ").title()}\nL2={l2:.4f}, TV={tv:.3f}',
                 fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('u')
    ax.legend(fontsize=7); ax.set_ylim(-0.3, 1.4)
    ax.grid(True, alpha=0.3)

axes_flat[-1].axis('off')
axes_flat[-1].text(0.5, 0.5,
    'Observations:\n\n'
    'Upwind: smears both step and Gaussian\n\n'
    'Lax-Wendroff: sharp Gaussian but\noscillates behind step\n\n'
    'Minmod: conservative — no oscillations,\nslightly smears\n\n'
    'Van Leer: good balance — smooth,\nno oscillations\n\n'
    'Superbee: sharpest step, but can\nbe over-compressive on Gaussian',
    ha='center', va='center', fontsize=9, transform=axes_flat[-1].transAxes,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle(f'Scheme comparison: CFL={c*dt/dx:.2f}, t={T}', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| Property | Upwind | Lax-Wendroff | TVD (Van Leer) |
|----------|--------|-------------|----------------|
| Order (smooth) | 1st | 2nd | 2nd |
| Stability | $\nu \leq 1$ | $\nu \leq 1$ | $\nu \leq 1$ |
| Near discontinuities | Diffusive (smears) | Dispersive (oscillates) | ✅ Non-oscillatory |
| Conserves TV? | ✅ Yes | ❌ No | ✅ Yes |
| Nonlinear? | No | No | ✅ Yes (by design) |

**OpenFOAM scheme names:**
- `linear`: central — 2nd order, may oscillate (use with care)
- `upwind`: 1st order upwind — safe but diffusive
- `linearUpwind`: 2nd order upwind — good for smooth flows
- `limitedLinear`: Van Leer-like TVD — recommended default for convection
- `Gauss limitedLinear 1`: OpenFOAM syntax for TVD convection scheme

---
**Next:** Module 3.5 — Multigrid Methods: why Gauss-Seidel stalls on smooth errors and how V-cycles fix it in $O(N)$ operations.